In [2]:
import sys
from pathlib import Path
import json
import requests


print(
    f"Python Version: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
)
print(f"Environment: {sys.executable}")

# Find project root and add to Python path
current_dir = Path.cwd()
if current_dir.name == "notebooks":
    project_root = current_dir.parent
elif (current_dir / "compose.yml").exists():
    project_root = current_dir
else:
    project_root = None

if project_root and (project_root / "compose.yml").exists():
    print(f"Project root: {project_root}")
    sys.path.insert(0, str(project_root))
else:
    print("Missing compose.yml - check directory")
    exit()

Python Version: 3.12.12
Environment: /Users/surekha/Documents/projects/RAG/research-assistant/.venv/bin/python
Project root: /Users/surekha/Documents/projects/RAG/research-assistant


In [3]:
from src.services.opensearch.factory import make_opensearch_client
from opensearchpy import OpenSearch

opensearch_client = make_opensearch_client()
opensearch_client.host = "http://localhost:9200"
opensearch_client.client = OpenSearch(hosts=["http://localhost:9200"],http_compress=True,use_ssl=False,verify_cert=False,ssl_assert_hostname=False,ssl_show_warn=False)

print(f"Client configured with host: {opensearch_client.host}")
print(f"Index name: {opensearch_client.index_name}")

is_healthy = opensearch_client.health_check()
if is_healthy:
    print("OpenSearch health check: PASSED")
else:
    print("OpenSearch health check: FAILED")

Client configured with host: http://localhost:9200
Index name: arxiv-papers
OpenSearch health check: PASSED


In [4]:
# Display Index configuration

from src.services.opensearch.index_config import ARXIV_PAPERS_INDEX,ARXIV_PAPERS_MAPPING

print(f"Index Name: {ARXIV_PAPERS_INDEX}")

properties = ARXIV_PAPERS_MAPPING["mappings"]["properties"]

for field_name,config in properties.items():
    field_type = config.get("type")
    analyzer = config.get("analyzer","")
    if analyzer:
        print(f"    . {field_name}: {field_type} [{analyzer}]")
    else:
        print(f"    . {field_name}: {field_type}")

Index Name: arxiv-papers
    . arxiv_id: keyword
    . title: text [text_analyzer]
    . authors: text [standard_analyzer]
    . abstract: text [text_analyzer]
    . categories: keyword
    . raw_text: text [text_analyzer]
    . pdf_url: keyword
    . published_date: date
    . created_at: date
    . updated_at: date


In [5]:
try:
    index_exists = opensearch_client.client.indices.exists(index=opensearch_client.index_name)

    if index_exists:
        print(f" Index {opensearch_client.index_name} already exists")

        stats = opensearch_client.get_index_stats()
        if stats and 'error' not in stats:
            print(f"    Document: {stats.get('document_count',0)}")
            print(f"    Size:{stats.get('size_in_bytes',0):,} bytes")
    else:
        print(f"Creating new index: {opensearch_client.index_name}")
        success = opensearch_client.create_index()

        if success:
            print(f"    Index created successfully")
        else:
            print(f"    Index creation failed")

except Exception as e:
    print(f"    Error with index management")

 Index arxiv-papers already exists
    Document: 120
    Size:371,534 bytes


### Data Pipeline - Airflow run 

The paper ingestion pipeline automatically
1. Fetches papers from arxiv API.
2. Stores papers in PostgreSQL
3. Indexes papers into OpenSearch

In [6]:
stats = opensearch_client.get_index_stats()

if stats and 'error' not in stats:
    doc_count = stats.get('document_count',0)

    if doc_count>0:
        print(f"  Success! Found {doc_count} documents in OpenSearch")

        sample = opensearch_client.search_papers("*", size=3)
        if sample.get("hits"):
            print("\n Sample papers")

            for i, paper in enumerate(sample['hits'],1):
                title = paper.get('title', 'Unknown')[:60]
                print(f"    {i}. {title}...")
    else:
        print(" No documents in OpenSearch yet")
        print("\nPlease run the Airflow DAG first (see instructions above)")
else:
    print("Could not retrieve index stats")

  Success! Found 120 documents in OpenSearch


In [8]:
# Simple BM25 Search
print("SIMPLE BM25 SEARCH")
print("=" * 40)

# Change this to any word from your papers
search_term = "artificial intelligence"  # Try different terms!

print(f"Searching for: '{search_term}'\n")

results = opensearch_client.search_papers(query=search_term, size=5)

if results.get("hits"):
    print(f"Found {results.get('total', 0)} total matches\n")

    for i, paper in enumerate(results["hits"], 1):
        print(f"{i}. {paper.get('title', 'Unknown')[:70]}...")
        print(f"   Score: {paper.get('score', 0):.2f}")
        print(f"   arXiv ID: {paper.get('arxiv_id', 'N/A')}\n")
else:
    print("No results found. Try searching for:")
    print("  • 'neural', 'model', 'algorithm'")
    print("  • Use '*' to see all papers")

SIMPLE BM25 SEARCH
Searching for: 'artificial intelligence'

Found 15 total matches

1. Harnessing Photonics for Machine Intelligence...
   Score: 16.98
   arXiv ID: 2604.10841v1

2. Error-free Training for MedMNIST Datasets...
   Score: 14.81
   arXiv ID: 2604.18916v1

3. Regulating Artificial Intimacy: From Locks and Blocks to Relational Ac...
   Score: 13.72
   arXiv ID: 2604.18893v1

4. Crowdsourcing of Real-world Image Annotation via Visual Properties...
   Score: 12.56
   arXiv ID: 2604.14449v1

5. Speaking to No One: Ontological Dissonance and the Double Bind of Conv...
   Score: 12.24
   arXiv ID: 2604.10833v1



In [9]:
query = {
    "query":{
        "match":{
            "title": "machine learning"
        }
    },
    "size":3
}

response = opensearch_client.client.search(
    index=opensearch_client.index_name, body=query
)

print(response)

print(f"Found {response["hits"]["total"]["value"]} results\n")
for hit in response["hits"]["hits"]:
    print(f"Title: {hit["_source"]["title"][:70]}...")

{'took': 63, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 14, 'relation': 'eq'}, 'max_score': 5.0633955, 'hits': [{'_index': 'arxiv-papers', '_id': '2604.18862v1', '_score': 5.0633955, '_source': {'arxiv_id': '2604.18862v1', 'title': 'Human-Machine Co-Boosted Bug Report Identification with Mutualistic Neural Active Learning', 'authors': 'Guoming Long, Shihai Wang, Hui Fang, Tao Chen', 'abstract': 'Bug reports, encompassing a wide range of bug types, are crucial for maintaining software quality. However, the increasing complexity and volume of bug reports pose a significant challenge in sole manual identification and assignment to the appropriate teams for resolution, as dealing with all the reports is time-consuming and resource-intensive. In this paper, we introduce a cross-project framework, dubbed Mutualistic Neural Active Learning (MNAL), designed for automated and more effective identification of bug reports f

In [11]:
query = {
    "query":{
        "multi_match": {
            "query": "AI Agents",
            "fields": ["title^2", "abstract", "authors"],
            "type": "best_fields"
        }
    },
    "size": 3
}

response = opensearch_client.client.search(index=opensearch_client.index_name, body=query)

print(f"Found {response["hits"]["total"]["value"]} results\n")

for hit in response["hits"]["hits"]:
    print(f"Title: {hit["_source"]["title"][:70]}...")
    print(f"Score: {hit["_score"]:.2f}")
    print(f"Authors: {', '.join(hit["_source"]["authors"][:2])}...\n")

Found 42 results

Title: AIBuildAI: An AI Agent for Automatically Building AI Models...
Score: 10.15
Authors: R, u...

Title: How Adversarial Environments Mislead Agentic AI?...
Score: 9.19
Authors: Z, h...

Title: Subliminal Transfer of Unsafe Behaviors in AI Agent Distillation...
Score: 8.73
Authors: J, a...



In [14]:
# Boosting Query

query = {
    "query":{
        "boosting":{
            "positive":{
                "match":{
                    "abstract":"deep learning"
                }
            },
            "negative":{
                "match":{
                    "abstract":"multimodel"
                }
            },
            "negative_boost": 0.1
        }
    },
    "size":3
}

response = opensearch_client.client.search(index=opensearch_client.index_name,body=query)

print(f"Query: Boost 'deep learning', demote 'survey' papers\n")
print(f"Found {response["hits"]["total"]["value"]} results\n")

for hit in response["hits"]["hits"]:
    title = hit["_source"]["title"][:70]
    abstract_snippet = hit["_source"]["abstract"][:100]
    print(f"Title: {title}...")
    print(f"Score: {hit["_score"]:.2f}")
    print(f"Abstract: {abstract_snippet}...\n")

Query: Boost 'deep learning', demote 'survey' papers

Found 49 results

Title: Robustness Analysis of Machine Learning Models for IoT Intrusion Detec...
Score: 4.99
Abstract: Ensuring the reliability of machine learning-based intrusion detection systems remains a critical ch...

Title: Modality-Aware and Anatomical Vector-Quantized Autoencoding for Multim...
Score: 4.34
Abstract: Learning a robust Variational Autoencoder (VAE) is a fundamental step for many deep learning applica...

Title: EAGLE: Edge-Aware Graph Learning for Proactive Delivery Delay Predicti...
Score: 4.24
Abstract: Modern logistics networks generate rich operational data streams at every warehouse node and transpo...



In [16]:
query = {
    "query":{
        "bool": {
            "must": [
                {
                    "match":{
                        "abstract": "neural"
                    }
                }
            ],
            "filter":[
                {
                    "terms":{
                        "categories":["cs.AI"]
                    }
                }
            ]
        }
    },
    "size": 3
}

response = opensearch_client.client.search(index=opensearch_client.index_name,body=query)

print(f"Found {response["hits"]["total"]["value"]} results\n")

for hit in response["hits"]["hits"]:
    title = hit["_source"]["title"][:70]
    categories = ', '.join(hit["_source"]['categories'])
    print(f"Title {title}...")
    print(f"Categories: {categories}")
    print(f"Score: {hit["_score"]:.2f}\n")

Found 13 results

Title Gradient-Based Program Synthesis with Neurally Interpreted Languages...
Categories: cs.LG, cs.AI
Score: 3.97

Title Human-Machine Co-Boosted Bug Report Identification with Mutualistic Ne...
Categories: cs.SE, cs.AI
Score: 3.71

Title Beyond Uniform Sampling: Synergistic Active Learning and Input Denoisi...
Categories: cs.LG, cs.AI
Score: 3.43



In [17]:
query = {
    "query": {
        "match_all":{}
    },
    "sort":[
        {
            "published_date":{
                "order": "desc"
            }
        }
    ],
    "size":5
}

response = opensearch_client.client.search(index=opensearch_client.index_name, body=query)

print(f"Query: All papers sorted by publication date (newest first)\n")

for hit in response["hits"]["hits"]:
    title = hit["_source"]["title"][:70]
    pub_date = hit["_source"]["published_date"][:10]
    print(f"Date: {pub_date} | {title}...")

Query: All papers sorted by publication date (newest first)

Date: 2026-04-20 | Error-free Training for MedMNIST Datasets...
Date: 2026-04-20 | MORPHOGEN: A Multilingual Benchmark for Evaluating Gender-Aware Morpho...
Date: 2026-04-20 | Gradient-Based Program Synthesis with Neurally Interpreted Languages...
Date: 2026-04-20 | Harmful Intent as a Geometrically Recoverable Feature of LLM Residual ...
Date: 2026-04-20 | Regulating Artificial Intimacy: From Locks and Blocks to Relational Ac...


In [18]:
# complex search with multiple criteria

query = {
    "query":{
        "bool":{
            "must":[
                {
                    "multi_match":{
                        "query": "transformer",
                        "fields": ["title","abstract"],
                        "type":"best_fields"
                    }
                }
            ],
            "filter": [
                {
                    "range":{
                        "published_date":{
                            "gte": "2024-01-01"
                        }
                    }
                }
            ],
            "should":[
                {
                    "match":{
                        "categories": "cs.AI"
                    }
                }
            ]
        }
    },
    "sort":[
        "_score", {"published_date": {"order": "desc"}}
    ],
    "size":3
}

response = opensearch_client.client.search(index=opensearch_client.index_name, body=query)

print(f"Found {response["hits"]["total"]["value"]} results\n")

for hit in response["hits"]["hits"]:
    title = hit["_source"]["title"][:70]
    pub_date = hit["_source"]["published_date"][:10]
    score = hit["_score"]
    categories = ", ".join(hit["_source"]["categories"][:2])

    print(f"Title: {title}...")
    print(f"Date: {pub_date} | score: {score:.2f}")
    print(f"Categories: {categories} \n")

Found 15 results

Title: Three-Phase Transformer...
Date: 2026-04-15 | score: 4.52
Categories: cs.CL, cs.AI 

Title: Hierarchical vs. Flat Iteration in Shared-Weight Transformers...
Date: 2026-04-15 | score: 3.61
Categories: cs.CL, cs.AI 

Title: Seeing Through Circuits: Faithful Mechanistic Interpretability for Vis...
Date: 2026-04-15 | score: 3.43
Categories: cs.AI 

